# A notebook to compile counts for the HRApop paper

# Import libraries

In [ ]:
%pip install pandas numpy requests 

import pandas as pd
import numpy as np
import requests
import io
from pprint import pprint

# Set global variables

In [ ]:
hra_pop_version = "v1.1"
branch = 'main'

# Load data

In [ ]:
sankey = pd.read_csv(
    f"https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/universe-ad-hoc/sankey.csv")

sankey

In [ ]:
# unique cells
universe_sc_transcriptomics_cell_counts = pd.read_csv(
    f'https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/universe-ad-hoc/universe-sc-transcriptomics-cell-counts.csv', index_col=False)
universe_sc_proteomics_cell_counts = pd.read_csv(
    f'https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/universe-ad-hoc/universe-sc-proteomics-cell-counts.csv', index_col=False)
universe_sc_transcriptomics_cell_instance_counts = pd.read_csv(
    f'https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/universe-ad-hoc/universe-sc-transcriptomics-cell-instance-counts.csv', index_col=False)

# Pre-processing steps

## Simplify rows with multiple annotations for getting accurate counts

In [ ]:
tool_replacement = "sc_transcriptomics with Cell Summary"

sankey['cell_type_annotation_tool'] = sankey['cell_type_annotation_tool'].replace({
    'azimuth': tool_replacement,
    'celltypist': tool_replacement,
    'popv': tool_replacement,
    np.nan: "No Cell Summary"
})

## Manually fix missing cell type annotation values for SenNet atlas datasets

See GitHub issue: https://github.com/x-atlas-consortia/hra-pop/issues/91

In [ ]:
# Define the indexing criteria
criteria = (sankey['portal'] == "SenNet") & (
    sankey['is_atlas_dataset'] == True)

# Apply the change to the SenNet atlas datasets (2 as of HRApop v0.10.2)
sankey.loc[criteria, 'cell_type_annotation_tool'] = tool_replacement

sankey = sankey.drop_duplicates()

# Get counts for HRApop paper

The following sections provide counts of datasets and other metrics for HRApop v0.10.2.

## Report numbers for Highlights

In [ ]:
# All datasets downloaded and retrieved from extraction sites
all_datasets = sankey['unique_dataset_id'].unique()

print(f"Number of UNIVERSE datasets: {len(all_datasets)}")

In [ ]:
# All sc-proteomics
all_sc_proteomics = sankey[['dataset_id', 'cell_type_annotation_tool']
                           ].loc[sankey['cell_type_annotation_tool'] == "sc_proteomics"].drop_duplicates()

print(f"Number of sc-proteomics datasets: {len(all_sc_proteomics)}")

In [ ]:
print(
    f'Number of sc-transcriptomics datasets: {len(sankey[(sankey['is_atlas_dataset'] == True) & (sankey['cell_type_annotation_tool'] != 'sc_proteomics')].drop_duplicates(subset=['unique_dataset_id']))}')

In [ ]:
# All datasets with cell summary
sc_transcriptomics_with_cell_summary = sankey[['unique_dataset_id', 'cell_type_annotation_tool']].loc[
    sankey['cell_type_annotation_tool'] == tool_replacement]['unique_dataset_id'].drop_duplicates()

print(f"Number of sc-transcriptomics datasets with cell summary: {len(sc_transcriptomics_with_cell_summary)}")

In [ ]:
# Organs in HRApop Atlas
organs_in_hra_pop = sankey.loc[sankey['is_atlas_dataset']
                                == True]['organ_name'].unique()
print(
    f"Unique organs in HRApop Atlas: {len(organs_in_hra_pop)}")

In [ ]:
# Organs (m/f) in HRApop Atlas
organs_in_hra_pop_sex = sankey.loc[sankey['is_atlas_dataset']
                               == True]['organ_name_glb_file'].unique()
print(
    f"Organs (m/f) in HRApop Atlas: {len(organs_in_hra_pop_sex)}")

In [ ]:
# Volume covered by HRApop tissue blocks
volume = sankey.loc[sankey['is_atlas_dataset']== True].drop_duplicates(subset=['unique_dataset_id'])['tissue_block_volume'].sum()
print(
    f"Volume covered by HRApop tissue blocks: {volume}")

## Report numbers for Sankey/experimental data

In [ ]:
# atlas datasets
atlas = sankey.loc[sankey['is_atlas_dataset'] == True]['unique_dataset_id'].unique()
print(f"Atlas datasets: {len(atlas)}\n")

# datasets with extraction site but without cell summary
no_cell_summary = sankey.loc[(sankey['is_rui_registered'] == True) & (
    sankey['cell_type_annotation_tool'] == "No Cell Summary")]['unique_dataset_id'].unique()
print(f"Datasets with extraction site but without cell summary: {
      len(no_cell_summary)}\n")

# datasets with cell summary but without extraction site
no_rui = sankey.loc[(sankey['is_rui_registered'] ==False) & (
    sankey['cell_type_annotation_tool'] != "No Cell Summary")]
print(f"Datasets with cell summary but without extraction site: {
      len(no_rui)}\n")

# datasets with cell summary 
cell_summary = sankey.loc[sankey['cell_type_annotation_tool'] != "No Cell Summary"]
print(f"Datasets with cell summary: {
      len(cell_summary)}\n")

# datasets with neither
non_atlas_without_either = sankey.loc[(sankey['cell_type_annotation_tool'] == "No Cell Summary") & (sankey['is_rui_registered'] == False)]
print(f"Datasets with neither: {len(non_atlas_without_either)}\n")

# non-atlas datasets total
non_atlas_total = sankey.loc[(
    sankey['is_atlas_dataset'] == False)]['unique_dataset_id'].unique()
print(f"Non-atlas datasets total: {len(non_atlas_total)}\n")

# unique cells
sc_transcriptomics_cell_counts = universe_sc_transcriptomics_cell_counts[
    'universe_sc_transcriptomics_cell_count'].iloc[0]
print(
    f'Unique cells from sc-transcriptomics datasets in the Universe: {sc_transcriptomics_cell_counts}\n')

sc_transcriptomics_cell_counts_preannotated = universe_sc_transcriptomics_cell_counts[
    'universe_sc_transcriptomics_preannotated_cell_count'].iloc[0]
print(
    f'Unique cells from sc-transcriptomics datasets in the Universe (preannotated): {sc_transcriptomics_cell_counts_preannotated}\n')

sc_proteomics_cell_counts = universe_sc_proteomics_cell_counts[
    'universe_sc_proteomics_cell_count'].iloc[0]
print(
    f'Unique cells from sc-proteomics datasets in the Universe: {sc_proteomics_cell_counts}\n')

In [ ]:
# h5ad files
print(f'Unique h5ad files: {sankey['link_to_h5ad_file'].nunique()}')

## AS Counts

In [ ]:
# Read the CSV data
df_as_data = pd.read_csv(
    f'https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/atlas-ad-hoc/cell-types-in-anatomical-structurescts-per-as.csv')

# Display the DataFrame
df_as_data

In [ ]:
# unique AS
print(f'Number of unique AS IDs in HRApop {hra_pop_version}: {len(df_as_data['as_label'].unique())}')

In [ ]:
unique_as_by_sex = df_as_data[['as_label', 'sex']].drop_duplicates()
print(
    f'Number of unique AS in HRApop {hra_pop_version} separated by sex: {len(unique_as_by_sex)}')

In [ ]:
print(f'Number of organs covered by sc-transcriptomics: {sankey[(sankey['is_atlas_dataset'] == True) & (sankey['cell_type_annotation_tool'] != 'sc_proteomics')]['organ_name'].nunique()}')

In [ ]:
print(f'Number of AS covered by sc-proteomics: {df_as_data[df_as_data['tool'] == 'sc_proteomics']['as_label'].nunique()}')

## Get counts for HRA 10th release

In [ ]:
sankey.groupby(['organ_name', 'is_atlas_dataset'])[
    'unique_dataset_id'].nunique().reset_index()

## Crosswalks

In [ ]:
crosswalk_azimuth = pd.read_csv(
    'https://cdn.humanatlas.io/digital-objects/ctann/azimuth/v1.2/assets/azimuth-crosswalk.csv', skiprows=10)
crosswalk_azimuth

In [ ]:
crosswalk_celltypist = pd.read_csv(
    'https://cdn.humanatlas.io/digital-objects/ctann/celltypist/v1.1/assets/celltypist-crosswalk.csv', skiprows=10)
crosswalk_celltypist

In [ ]:
crosswalk_popv = pd.read_csv(
    'https://cdn.humanatlas.io/digital-objects/ctann/popv/v1.2/assets/popv-crosswalk.csv', skiprows=10)
crosswalk_popv

In [ ]:
crosswalk_vccf = pd.read_csv(
    'https://cdn.humanatlas.io/digital-objects/ctann/vccf/v1.0/assets/vccf-crosswalk.csv', skiprows=10)
crosswalk_vccf

In [ ]:
# extract CL IDs and labels with tool
extract = ['Organ_ID','Annotation_Label','Annotation_Label_ID','CL_ID', 'CL_Label', 'CL_Match']

# Extract the columns from each DataFrame
az_selected = crosswalk_azimuth[extract].assign(tool='azimuth')
ct_selected = crosswalk_celltypist[extract].assign(tool='celltypist')
popv_selected = crosswalk_popv[extract].assign(tool='popv')
vccf_selected = crosswalk_vccf[extract].assign(tool='vccf')

# Concatenate them into one DataFrame
df_crosswalks_combined = pd.concat(
    [az_selected, ct_selected, popv_selected, vccf_selected], ignore_index=True)

df_crosswalks_combined

In [ ]:
# get counts
print(f'Unique values across all crosswalks: {df_crosswalks_combined.nunique()}')

In [ ]:
# types of matches
df_crosswalks_combined.groupby('CL_Match').size()

In [ ]:
df_crosswalks_combined.drop_duplicates(subset=['Annotation_Label_ID','CL_ID', 'tool'])

In [ ]:
print(f'Number unique crosswalking operations: {len(df_crosswalks_combined)}')

In [ ]:
# not crosswalked
df_not_crosswalked = pd.read_csv(
    f'https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/atlas-ad-hoc/unmapped-cell-ids.csv')
df_not_crosswalked

In [ ]:
print(
    f'Number of cell labels from CTann tools and sc-proteomics data that were not crosswalked: {df_not_crosswalked['cell_label'].nunique()}')

In [ ]:
# number of cell IDs aggregated tp higher levels
df_level_1_2 = pd.read_csv(
    f'https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/atlas-ad-hoc/cell-types-level-mapping.csv')
df_level_1_2

In [ ]:
print(
    f'Number of unique  cell IDs aggregated to higher levels: {df_level_1_2['cell_label'].nunique()}')
print(
    f'Number of unique cell IDs in level 1: {df_level_1_2['level_1_cell_label'].nunique()}')
print(
    f'Number of unique cell IDs in level 2: {df_level_1_2['level_2_cell_label'].nunique()}')

In [ ]:
# CTs per organ per tool from crosswalks. Alt source: https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/105da2e49b1b10d6531fb0fc302bd4cbb3c197eb/output-data/v1.0/reports/hra/ct-per-organ-per-tool.csv


In [ ]:
# from crosswalks
df_crosswalks_combined.groupby(['tool','CL_ID']).size()

In [ ]:
df_crosswalks_combined

In [ ]:
tally = (
    df_crosswalks_combined.groupby(['Organ_ID', 'tool'])['CL_Label']
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)

def try_get_organ_label(id:str):
  """_summary_

  Args:
      id (str): _description_
  """
  
  # Taken from https://github.com/hubmapconsortium/hra-workflows-runner/blob/main/src%2Fgtex%2Fdownloader.js#L24-L39
  ORGAN_MAPPING = {
      "bladder": "UBERON:0001255",
      "blood": "UBERON:0000178",
      "bone_marrow": "UBERON:0002371",
      "eye": "UBERON:0000970",
      "heart": "UBERON:0000948",
      "large_intestine": "UBERON:0000059",
      "liver": "UBERON:0002107",
      "lung": "UBERON:0002048",
      # or mesenteric lymph node (UBERON:0002509)?
      "lymph_node": "UBERON:0000029",
      "mammary": "UBERON:0001911",
      "pancreas": "UBERON:0001264",
      "prostate": "UBERON:0002367",
      "skin": "UBERON:0002097",
      "small_intestine": "UBERON:0002108",
      "spleen": "UBERON:0002106",
      "thymus": "UBERON:0002370",
      "trachea": "UBERON:0003126",
      "uterus": "UBERON:0000995",
      "vasculature": "UBERON:0004537",
      "breast": "UBERON:0001911",
      "esophagus mucosa": "UBERON:0002469",
      "esophagus muscularis": "UBERON:0004648",
      "skeletal muscle": "UBERON:0001134",
  }
  
  try:
    value_to_find = id
    keys = [k for k, v in ORGAN_MAPPING.items() if v == value_to_find]
   # Output: ['a', 'c']
    return keys
  except:
    return ""
  

tally['Organ_Label'] = tally['Organ_ID'].apply(lambda id: try_get_organ_label(id))
tally

In [ ]:
def get_uberon_label(uberon_id: str) -> str:
    """Fetch the label for a given UBERON ID using the OLS API."""
    base_url = "https://www.ebi.ac.uk/ols/api/ontologies/uberon/terms"
    iri = f"http://purl.obolibrary.org/obo/{uberon_id.replace(':', '_')}"

    response = requests.get(base_url, params={"iri": iri})
    print("Request URL:", response.url)

    if response.ok:
        data = response.json()
        terms = data.get("_embedded", {}).get("terms", [])
        if terms:
            return terms[0].get("label", "Label not found")
        else:
            return "Label not found in response"
    else:
        return f"Error: {response.status_code}"


# Example
label = get_uberon_label("UBERON:0002450")
print("Label:", label)

# Random queries

In [ ]:
# get ATLAS datasets with donors < 18
underage = sankey[(sankey['donor_age'] < 18) & (sankey['is_atlas_dataset'] == True)]
underage

In [ ]:
sankey

In [ ]:
# dataset IDs for sc-proteomics
dois = sankey[sankey['cell_type_annotation_tool'] =='sc_proteomics']
dois['doi'].unique()

In [ ]:
# numbers for 10th 2/3D Datasets: https://docs.google.com/spreadsheets/d/1xG4stdTZW37pmgX4kAMOnsqtjHv_tbiDbGET2Tcokuk/edit?gid=1213346061#gid=1213346061
print(f'Universe datasets for brain: ')

sankey[sankey['organ_name'].str.contains('brain', na=False)]

sankey[(sankey['organ_name'].str.contains('lymph', na=False)) & (sankey['is_atlas_dataset'] == False)]['unique_dataset_id'].nunique()

In [ ]:
# 16,293 datasets -- all human & healthy, all ages. For how many can we do US1/2 prediction?
df_us1 = pd.read_csv(f'https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/atlas/application-a1.csv')

#US#2: most similar AS
df_us2_as = pd.read_csv(
    f'https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/atlas/application-a2p1.csv')

#US#2: most similar extraction site
df_us2_es = pd.read_csv(
    f'https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/atlas/application-a2p3.csv')

In [ ]:
df_us1

In [ ]:
df_us2_as

In [ ]:
df_us2_es

In [ ]:
print(f'Unique extraction sites for which we predict US#1: {df_us1['rui_location'].nunique()}.')

In [ ]:
print(f'Unique datasets for US#2 (most similar AS): {df_us2_as['dataset'].nunique()}.')

In [ ]:
print(
    f'Unique datasets for US#2 (most similar extraction site/corridor): {df_us2_es['dataset'].nunique()}.')

In [ ]:
# How many are adult?
sankey[sankey['donor_age'] >= 18]['unique_dataset_id'].nunique()

## SenNet marker paper

In [ ]:
df_sennet = sankey[(sankey["portal"] == "SenNet")]
df_sennet

In [ ]:
df_sennet[(df_sennet["is_atlas_dataset"] == False) & (df_sennet['cell_type_annotation_tool'] == 'sc_proteomics')]

In [ ]:
sankey.columns

## Extraction sites

In [ ]:
url = f'https://raw.githubusercontent.com/x-atlas-consortia/hra-pop/refs/heads/{branch}/output-data/{hra_pop_version}/reports/universe-ad-hoc/extraction-sites.csv'

universe_extraction_sites = pd.read_csv(url)
universe_extraction_sites

In [ ]:
def get_etraction_site_and_mesh_collisions(iri:str):
  """Takes an IRI, gets the extraction site and mesh collisions

  Args:
      iri (str): IRI for the extraction site
  """
  # initialize result
  result = (set(), set())
  
  # loop through extraction site IDs and get extraction site data
  api_extraction_site_base = 'https://apps.humanatlas.io/api/v1/extraction-site?iri='
  api_collisions_base = 'https://apps.humanatlas.io/api/v1/collisions'
  
  try:
    response = requests.get(api_extraction_site_base+iri)
    if response.ok:
        print(f'Successfully got extraction site data for {iri}!')
        extraction_site = response.json()
        try:
          headers = {
              "accept": "application/json",
              "content-type": "application/json"
          }
          data = extraction_site
          response = requests.post(api_collisions_base, headers=headers, json=data)
          if response.ok:
              print(f'Successfully got mesh collisions for {extraction_site['@id']}!')
              mesh = response.json()
              mesh_iris = [collision['representation_of'] for collision in mesh]
              result[0].update(mesh_iris)
              
              organ_iris = [collision['organ'] for collision in mesh]
              result[1].update(organ_iris)
              
          else:
              print(f"Request failed with status code {response.status_code}")
        except requests.exceptions.RequestException as e:
          print(f"An error occurred: {e}")
    else: 
      print(f"Request failed with status code {response.status_code}")
  except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")
    
  print(f"Returning {result}")
  print()
  return result

In [ ]:
unique_iris = {
    'anatomical_structures': set(),
    'organs': set()
}

# Apply the function to each row
results = universe_extraction_sites['extraction_site'].apply(
    lambda iri: get_etraction_site_and_mesh_collisions(iri)
)

# Unpack the tuple of sets and update each unique set
for result in results:
    print(f'now working with {result}')
    if isinstance(result, tuple) and len(result) == 2:
        as_set, organ_set = result
        unique_iris['anatomical_structures'].update(as_set)
        unique_iris['organs'].update(organ_set)

# Print results
pprint(unique_iris)

In [ ]:
print(
    f'Number of AS covered across extraction sites in Universe: {len(unique_iris['anatomical_structures'])}')

print(
    f'Number of organs covered across extraction sites in Universe: {len(unique_iris['organs'])}')